In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = ""
from pathlib import Path
import Auxiliares.Funciones.Boards.func_load_npz as loader
import numpy as np

In [2]:
seeds = [221, 502, 700, 1204, 3340, 4501, 6054, 6621, 8421, 15678,  19302, 38475, 77293, 91827, 99100]
base_dir = Path("./Resultados/Resultados_Classic/Boards")

for seed in seeds:
    # 1. Manejo seguro de rutas 
    archivo_test = base_dir/f"Test/test_{seed}.npz"
    directorio_train = base_dir/f"Train/Train_{seed}"
    
    print(f"\n--- Analizando Semilla {seed} ---")
    
    # 2. Cargar el archivo Test y crear el set de control
    hashes_test = set()
    if not archivo_test.exists():
        print(f"Error: Archivo de test no encontrado: {archivo_test}")
        continue
        
    _, test_boards, _ = loader.load_npz(archivo_test, archivo_test.name, log=False)
    test_boards_flat = test_boards.reshape(test_boards.shape[0], -1)
    
    for board in test_boards_flat:
        hashes_test.add(board.tobytes())
        
    print(f"  Tableros en Test cargados: {len(hashes_test):,}")
    del test_boards, test_boards_flat # Liberar memoria RAM
    
    # 3. Obtener archivos de Train
    archivos_train = [f for f in directorio_train.iterdir() if f.is_file() and f.suffix == '.npz']
    
    # Estructuras para el rastreo global
    tableros_vistos_global = set()
    total_tableros_leidos = 0
    duplicados_totales_train = 0
    fugas_hacia_test = 0  
    
    # 4. Ordenación nativa
    for archivo in sorted(archivos_train): 
        _, init_boards, _ = loader.load_npz(archivo, archivo.name, log=False)
        boards_flat = init_boards.reshape(init_boards.shape[0], -1)
        total_tableros_leidos += len(boards_flat)
        
        for board in boards_flat:
            b_hash = board.tobytes()
            
            # Comprobar Leakage (Fuga de datos)
            if b_hash in hashes_test:
                fugas_hacia_test += 1
            
            # Comprobar Duplicados internos de Train
            if b_hash in tableros_vistos_global:
                duplicados_totales_train += 1
            else:
                tableros_vistos_global.add(b_hash)
                
        # Liberar memoria del array cargado por cada iteración del archivo
        del init_boards, boards_flat

    # Report
    print("-" * 50)
    print(f"RESUMEN SEMILLA {seed}")
    print(f"Total de tableros procesados (Train): {total_tableros_leidos:,}")
    print(f"Tableros únicos reales (Train):       {len(tableros_vistos_global):,}")
    print(f"Tableros duplicados en Train:         {duplicados_totales_train:,} ({(duplicados_totales_train/total_tableros_leidos)*100:.2f}%)")
    
    print(f"Fugas en Test (Data Leakage):         {fugas_hacia_test:,}")


--- Analizando Semilla 221 ---
  Tableros en Test cargados: 15,000
--------------------------------------------------
RESUMEN SEMILLA 221
Total de tableros procesados (Train): 1,872,000
Tableros únicos reales (Train):       1,871,774
Tableros duplicados en Train:         226 (0.01%)
Fugas en Test (Data Leakage):         5

--- Analizando Semilla 502 ---
  Tableros en Test cargados: 15,000
--------------------------------------------------
RESUMEN SEMILLA 502
Total de tableros procesados (Train): 1,824,000
Tableros únicos reales (Train):       1,823,780
Tableros duplicados en Train:         220 (0.01%)
Fugas en Test (Data Leakage):         4

--- Analizando Semilla 700 ---
  Tableros en Test cargados: 15,000
--------------------------------------------------
RESUMEN SEMILLA 700
Total de tableros procesados (Train): 1,872,000
Tableros únicos reales (Train):       1,871,755
Tableros duplicados en Train:         245 (0.01%)
Fugas en Test (Data Leakage):         5

--- Analizando Semilla 1